# Módulo 08 · Method chaining y pipelines

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Escribir transformaciones como **una cadena** legible de métodos, de arriba hacia abajo.
2. Crear columnas con **`assign`** (incluidas columnas que dependen de otras recién creadas).
3. Filtrar con **`query`** y calcular con **`eval`**, usando variables externas y nombres con espacios.
4. Insertar funciones propias en la cadena con **`pipe`**.
5. **Depurar** cadenas sin romperlas.
6. Organizar un análisis como un **pipeline por etapas**: funciones pequeñas, probables y reutilizables.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | ¿Por qué encadenar? |
| 2 | `assign` |
| 3 | `query` |
| 4 | `eval` |
| 5 | `pipe` |
| 6 | Otros métodos amigables con las cadenas |
| 7 | Depurar una cadena |
| 8 | Pipelines por etapas |
| 9 | Antipatrones |
| ✍️ | Ejercicios · ✅ Soluciones |

## 0 · Setup y carga

In [ ]:
from functools import wraps
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

In [ ]:
orders = pd.read_csv(
  DATA / "olist_orders_dataset.csv",
  dtype={"order_status": "category"},
  parse_dates=["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
)
items = pd.read_csv(DATA / "olist_order_items_dataset.csv", dtype={"order_item_id": "int8"},
                    parse_dates=["shipping_limit_date"])
products = pd.read_csv(DATA / "olist_products_dataset.csv")
traduccion = pd.read_csv(DATA / "product_category_name_translation.csv")
reviews = pd.read_csv(DATA / "olist_order_reviews_dataset.csv",
                      usecols=["review_id", "order_id", "review_score", "review_answer_timestamp"],
                      parse_dates=["review_answer_timestamp"])
customers = pd.read_csv(DATA / "olist_customers_dataset.csv",
                        usecols=["customer_id", "customer_state"], dtype={"customer_state": "category"})
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv",
                      usecols=["seller_id", "seller_state"], dtype={"seller_state": "category"})

## 1 · ¿Por qué encadenar?

Pregunta de negocio: **ingreso y flete promedio por categoría (en inglés), para pedidos entregados, top 5 por ingreso.**

### Estilo imperativo

In [ ]:
df = items.merge(orders[["order_id", "order_status"]], on="order_id")
df = df[df["order_status"] == "delivered"]
df = df.merge(products[["product_id", "product_category_name"]], on="product_id")
df = df.merge(traduccion, on="product_category_name", how="left")
df["categoria"] = df["product_category_name_english"].fillna("sin_categoria")
res = df.groupby("categoria").agg(ingreso=("price", "sum"), flete_medio=("freight_value", "mean"))
res = res.sort_values("ingreso", ascending=False)
res = res.head(5)
res_imperativo = res.round(2)
res_imperativo

Funciona, pero: la variable `df` cambia de significado 5 veces, es fácil ejecutar celdas fuera de orden en Jupyter y quedan variables intermedias en memoria.

### Estilo encadenado

In [ ]:
res_cadena = (
  items
  .merge(orders[["order_id", "order_status"]], on="order_id", validate="m:1")
  .query("order_status == 'delivered'")
  .merge(products[["product_id", "product_category_name"]], on="product_id", validate="m:1")
  .merge(traduccion, on="product_category_name", how="left", validate="m:1")
  .assign(categoria=lambda d: d["product_category_name_english"].fillna("sin_categoria"))
  .groupby("categoria")
  .agg(ingreso=("price", "sum"), flete_medio=("freight_value", "mean"))
  .sort_values("ingreso", ascending=False)
  .head(5)
  .round(2)
)
assert res_cadena.equals(res_imperativo)
res_cadena

Ventajas:

- Se **lee como una receta**: una línea, un paso.
- No hay estado intermedio que se pueda desincronizar: la celda es **reproducible** de una vez.
- Puedes comentar una línea (`# .head(5)`) para inspeccionar.
- Con Copy-on-Write (pandas 3), los pasos intermedios **no copian datos** innecesariamente.

La clave técnica: cada método devuelve un **objeto nuevo**, y los paréntesis exteriores permiten partir la expresión en varias líneas.

## 2 · `assign`

`assign(nueva=valor)` devuelve una **copia** con la columna nueva. El valor puede ser:

- una Series/array/escalar **calculado fuera** (usa el DataFrame original), o
- una **función** `lambda d: ...` que recibe el DataFrame **en ese punto de la cadena**. ✅ Preferida en cadenas.

In [ ]:
(items
 .assign(
   total=lambda d: d["price"] + d["freight_value"],
   pct_flete=lambda d: d["freight_value"] / d["total"] * 100,    # usa "total", creada en la línea anterior
   flete_caro=lambda d: d["pct_flete"] > 25,
 )
 [["price", "freight_value", "total", "pct_flete", "flete_caro"]]
 .head())

Los argumentos de un mismo `assign` se evalúan **en orden**, así que una columna puede usar las anteriores.

### ¿Por qué `lambda d:` y no `items["price"]`?

Porque dentro de una cadena el DataFrame **ya no es** `items`: puede estar filtrado o reordenado. Referenciar la variable original rompe la alineación o usa datos equivocados.

In [ ]:
# ❌ Mezcla el DataFrame filtrado con la Series original: se alinea por índice y deja NaN
mal = items.query("price > 500").assign(total=items["price"] + items["freight_value"])
# ✅ La lambda recibe el DataFrame filtrado
bien = items.query("price > 500").assign(total=lambda d: d["price"] + d["freight_value"])
print(len(mal), len(bien))
print("¿iguales?", mal["total"].equals(bien["total"]))

En este caso coincide por la alineación por índice (Módulo 02)... pero si el paso previo hubiera sido un `merge`, `reset_index` o `groupby`, el índice ya no correspondería y el resultado sería **incorrecto sin error**. La lambda evita pensar en eso.

### Nombres no válidos como identificadores

Para columnas con espacios o acentos, desempaqueta un diccionario:

In [ ]:
items.assign(**{"precio con flete": lambda d: d["price"] + d["freight_value"]}).iloc[:2, -2:]

## 3 · `query`

`query` filtra filas con una **expresión en texto**. Es la contraparte encadenable de `df[mascara]`.

| Necesito… | En `query` |
|---|---|
| comparar columnas | `"price > freight_value * 5"` |
| combinar condiciones | `and`, `or`, `not` (o `&`, `|`, `~`) |
| pertenencia | `"seller_state in ['SP', 'RJ']"` / `not in` |
| variable de Python | `"price > @umbral"` |
| columna con espacios | `` "`precio con flete` > 100" `` |
| métodos | `"order_purchase_timestamp.dt.year == 2018"`, `"product_category_name.isna()"` |
| rango | `"10 <= price < 50"` |

In [ ]:
umbral = 200
estados = ["SP", "RJ"]

(items
 .merge(sellers, on="seller_id", validate="m:1")
 .query("price > @umbral and seller_state in @estados")
 [["order_id", "price", "seller_state"]]
 .head())

In [ ]:
orders.query("order_purchase_timestamp.dt.year == 2018 and order_delivered_customer_date.isna()")[
  ["order_id", "order_status", "order_purchase_timestamp"]]

### `query` vs `loc[lambda d: ...]`

`query` es muy legible para condiciones simples. Para condiciones complejas (regex, funciones propias), `loc` con una lambda es igual de encadenable y no requiere aprender otra mini-sintaxis:

In [ ]:
a = products.query("product_category_name.str.startswith('moveis', na=False)")
b = products.loc[lambda d: d["product_category_name"].str.startswith("moveis", na=False)]
a.equals(b), len(a)

## 4 · `eval`

`eval` evalúa **expresiones aritméticas** sobre columnas. Útil para:

- Escribir fórmulas legibles (sin repetir `d["..."]`).
- Crear **varias columnas** de una vez con asignaciones en líneas separadas.
- Potencialmente, rendimiento en DataFrames **muy grandes** (usa `numexpr` si está instalado). Lo mediremos.

In [ ]:
(products
 .eval('''
     volumen_cm3 = product_length_cm * product_height_cm * product_width_cm
     densidad_g_cm3 = product_weight_g / volumen_cm3
 ''')
 [["product_weight_g", "volumen_cm3", "densidad_g_cm3"]]
 .describe()
 .round(3))

¿Es más rápido? Midámoslo en lugar de suponerlo, con ~600 mil filas:

In [ ]:
grande = pd.concat([products] * 2000, ignore_index=True)
print(f"{len(grande):,} filas")

In [ ]:
%timeit grande["product_weight_g"] / (grande["product_length_cm"] * grande["product_height_cm"] * grande["product_width_cm"])

In [ ]:
%timeit grande.eval("product_weight_g / (product_length_cm * product_height_cm * product_width_cm)")

Cuando preparé el curso, `eval` resultó **más lento** (≈5 ms contra ≈2.3 ms; tus números variarán según tu equipo): interpretar la expresión tiene un costo fijo y las operaciones vectorizadas de pandas ya son rápidas. `numexpr` sólo gana en expresiones largas sobre decenas de millones de filas (evita crear arreglos temporales), y aun así conviene medir. **Elige `eval` por legibilidad**, no por velocidad.

## 5 · `pipe`

`df.pipe(f, *args, **kwargs)` es lo mismo que `f(df, *args, **kwargs)`, pero **dentro de la cadena**. Así puedes insertar funciones propias (o de otras librerías) sin romper el flujo.

In [ ]:
def agregar_categoria_en(df: pd.DataFrame, traduccion: pd.DataFrame) -> pd.DataFrame:
  return (df
          .merge(products[["product_id", "product_category_name"]], on="product_id", how="left", validate="m:1")
          .merge(traduccion, on="product_category_name", how="left", validate="m:1")
          .rename(columns={"product_category_name_english": "categoria"})
          .drop(columns="product_category_name"))


def top_n(df: pd.DataFrame, por: str, n: int = 5) -> pd.DataFrame:
  return df.nlargest(n, por)


(items
 .pipe(agregar_categoria_en, traduccion)
 .groupby("categoria", as_index=False)
 .agg(ingreso=("price", "sum"))
 .pipe(top_n, por="ingreso", n=3))

### Cuando el DataFrame no es el primer argumento

Pasa una tupla `(funcion, "nombre_del_argumento")`:

In [ ]:
def resumen(titulo: str, datos: pd.DataFrame) -> pd.DataFrame:
  print(f"{titulo}: {len(datos)} filas")
  return datos

items.pipe((resumen, "datos"), titulo="Ítems cargados").shape

### `pipe` también existe en `GroupBy`

In [ ]:
(items
 .groupby("seller_id")
 .pipe(lambda g: g["price"].sum() / g["order_id"].nunique())   # ingreso por pedido de cada vendedor
 .sort_values(ascending=False)
 .head(3)
 .round(2))

## 6 · Otros métodos amigables con las cadenas

| Quiero… | Método encadenable |
|---|---|
| renombrar columnas | `rename(columns={...})`, `rename(columns=str.lower)`, `add_prefix`, `set_axis` |
| seleccionar columnas | `[[...]]`, `filter(like=..., regex=...)`, `drop(columns=...)`, `select_dtypes` |
| cambiar tipos | `astype({...})` |
| condicionales | `where`, `mask`, `clip`, `np.select` dentro de `assign` |
| ordenar / top | `sort_values`, `nlargest`, `nsmallest` |
| índice | `set_index`, `reset_index`, `rename_axis` |
| presentar | `style...` (siempre al **final**) |

In [ ]:
(products
 .filter(regex=r"^product_(weight|length|height|width)")
 .rename(columns=lambda c: c.removeprefix("product_").removesuffix("_cm").removesuffix("_g"))
 .pipe(lambda d: d.clip(upper=d.quantile(0.99), axis=1))   # recorta outliers al percentil 99 por columna
 .describe()
 .loc[["mean", "max"]]
 .round(1))

In [ ]:
(orders
 .assign(
   dias=lambda d: (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days,
   segmento=lambda d: np.select(
     [d["dias"].isna(), d["dias"] <= 7, d["dias"] <= 15],
     ["sin_entrega", "rapido", "normal"],
     default="lento",
   ),
 )
 ["segmento"].value_counts())

## 7 · Depurar una cadena

Tres técnicas, de menor a mayor esfuerzo:

1. **Comentar** las últimas líneas y ejecutar para ver el estado intermedio.
2. Insertar un **`pipe` de inspección** que imprime algo y devuelve el DataFrame **sin cambios**.
3. Partir la cadena en dos variables si un paso es muy complejo.

In [ ]:
def ver(df: pd.DataFrame, msg: str = "", n: int = 0) -> pd.DataFrame:
  print(f"[{msg}] filas={len(df):,} cols={df.shape[1]}")
  if n:
    display(df.head(n))
  return df


(items
 .pipe(ver, "items")
 .merge(orders[["order_id", "order_status"]], on="order_id", validate="m:1")
 .pipe(ver, "tras merge")
 .query("order_status == 'delivered'")
 .pipe(ver, "entregados")
 .groupby("seller_id")["price"].sum()
 .pipe(lambda s: print(f"[final] {len(s)} vendedores") or s)
 .nlargest(3))

Un decorador que registra **cada etapa** de un pipeline automáticamente (forma y tiempo), útil en código de producción:

In [ ]:
import time


def registrar(func):
  @wraps(func)
  def envoltura(df, *args, **kwargs):
    t0 = time.perf_counter()
    resultado = func(df, *args, **kwargs)
    ms = (time.perf_counter() - t0) * 1000
    print(f"{func.__name__:<22} {df.shape!s:>12} -> {resultado.shape!s:<12} {ms:6.1f} ms")
    return resultado
  return envoltura

## 8 · Pipelines por etapas

Para análisis que vas a repetir, organiza el trabajo en **funciones pequeñas con una sola responsabilidad**, cada una recibe y devuelve un DataFrame, y compónlas con `pipe`.

```
cargar → limpiar → enriquecer → filtrar → agregar → presentar
```

Beneficios: cada etapa se puede **probar** por separado, **reutilizar** y **reordenar**, y el pipeline completo se lee como un índice.

In [ ]:
@registrar
def limpiar_pedidos(df: pd.DataFrame) -> pd.DataFrame:
  return (df
          .dropna(subset=["order_purchase_timestamp"])
          .drop_duplicates("order_id"))


@registrar
def agregar_tiempos(df: pd.DataFrame) -> pd.DataFrame:
  return df.assign(
    dias_entrega=lambda d: (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.total_seconds() / 86400,
    tarde=lambda d: d["order_delivered_customer_date"] > d["order_estimated_delivery_date"],
    mes=lambda d: d["order_purchase_timestamp"].dt.to_period("M"),
  )


@registrar
def agregar_estado_cliente(df: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
  return df.merge(customers, on="customer_id", how="left", validate="1:1")


@registrar
def solo_entregados(df: pd.DataFrame) -> pd.DataFrame:
  return df.query("order_status == 'delivered' and order_delivered_customer_date.notna()")


@registrar
def kpis_por(df: pd.DataFrame, llave: str) -> pd.DataFrame:
  return (df.groupby(llave, observed=True)
            .agg(pedidos=("order_id", "size"),
                 dias_mediana=("dias_entrega", "median"),
                 pct_tarde=("tarde", "mean"))
            .assign(pct_tarde=lambda d: d["pct_tarde"] * 100))


kpis_estado = (
  orders
  .pipe(limpiar_pedidos)
  .pipe(agregar_tiempos)
  .pipe(agregar_estado_cliente, customers)
  .pipe(solo_entregados)
  .pipe(kpis_por, "customer_state")
  .query("pedidos >= 10")
  .sort_values("pct_tarde", ascending=False)
  .round(1)
)
kpis_estado

Cambiar el análisis es cambiar **una línea**: KPIs por mes en lugar de por estado, reutilizando las mismas etapas.

In [ ]:
(orders
 .pipe(limpiar_pedidos)
 .pipe(agregar_tiempos)
 .pipe(solo_entregados)
 .pipe(kpis_por, "mes")
 .tail(4)
 .round(1))

Y como cada etapa es una función pura, puedes **probarla** con datos mínimos:

In [ ]:
mini = pd.DataFrame({
  "order_id": ["a", "a", "b"],
  "order_purchase_timestamp": pd.to_datetime(["2018-01-01", "2018-01-01", None]),
})
assert limpiar_pedidos(mini)["order_id"].tolist() == ["a"]
print("✓ limpiar_pedidos pasa la prueba")

## 9 · Antipatrones

| ❌ Evita | ✅ Prefiere |
|---|---|
| `inplace=True` (no encadena, no ahorra memoria con CoW) | reasignar o encadenar |
| `df["a"][mask] = x` (asignación encadenada, no funciona en pandas 3) | `df.loc[mask, "a"] = x` o `assign` + `where`/`mask` |
| Referenciar la variable original dentro de `assign` | `lambda d: ...` |
| Cadenas de 40 líneas que hacen 6 cosas distintas | etapas con nombre y `pipe` |
| `apply(axis=1)` fila a fila dentro de la cadena | operaciones vectorizadas, `np.select`, `eval` |
| `style` en medio de la cadena | `style` sólo al final: devuelve un `Styler`, no un DataFrame |

In [ ]:
# Ejemplo: apply fila a fila vs vectorizado, ambos encadenables
lento = items.assign(total=lambda d: d.apply(lambda fila: fila["price"] + fila["freight_value"], axis=1))
rapido = items.assign(total=lambda d: d["price"] + d["freight_value"])
lento["total"].equals(rapido["total"])

---
# ✍️ Ejercicios

Variables y funciones disponibles: `orders`, `items`, `products`, `traduccion`, `reviews`, `customers`, `sellers`, `ver`, `registrar`.

### Ejercicio 1 · Refactorizar a una cadena ⭐

Reescribe este código como **una sola cadena** que produzca exactamente el mismo resultado en `res_e1` (sin variables intermedias):

```python
df = orders.copy()
df = df[df["order_status"] == "delivered"]
df["anio"] = df["order_purchase_timestamp"].dt.year
df["dias"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df = df[df["dias"].notna()]
df = df.merge(customers, on="customer_id")
res = df.groupby(["anio", "customer_state"], observed=True)["dias"].mean().reset_index()
res = res[res["customer_state"].isin(["SP", "RJ", "MG"])]
res = res.sort_values(["anio", "dias"], ascending=[True, False])
res = res.reset_index(drop=True)
```

In [ ]:
# Referencia (no la modifiques)
_df = orders.copy()
_df = _df[_df["order_status"] == "delivered"]
_df["anio"] = _df["order_purchase_timestamp"].dt.year
_df["dias"] = (_df["order_delivered_customer_date"] - _df["order_purchase_timestamp"]).dt.days
_df = _df[_df["dias"].notna()]
_df = _df.merge(customers, on="customer_id")
_ref = _df.groupby(["anio", "customer_state"], observed=True)["dias"].mean().reset_index()
_ref = _ref[_ref["customer_state"].isin(["SP", "RJ", "MG"])]
_ref = _ref.sort_values(["anio", "dias"], ascending=[True, False]).reset_index(drop=True)

# TU CÓDIGO AQUÍ
res_e1 = ...

In [ ]:
# Comprobación
if pendiente(res_e1):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  pd.testing.assert_frame_equal(res_e1, _ref, check_dtype=False, check_categorical=False)
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · `query` con variables y nombres raros ⭐

1. `ventas_q`: a partir de `items`, una cadena que:
   - cree la columna `"valor total"` (**con espacio**) = `price + freight_value`;
   - una `sellers` para tener `seller_state`;
   - con **un solo `query`**, conserve los ítems cuyo `"valor total"` esté **entre** `minimo` y `maximo` (ambos incluidos), cuyo estado esté en `estados_obj` y cuyo flete sea menor al 20% del precio.
2. `n_q`: número de filas resultantes.

In [ ]:
minimo, maximo = 50, 300
estados_obj = ["SP", "PR", "SC"]

# TU CÓDIGO AQUÍ
ventas_q = ...
n_q = ...

In [ ]:
# Comprobación
if pendiente(ventas_q, n_q):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert "valor total" in ventas_q.columns and "seller_state" in ventas_q.columns
  assert ventas_q["valor total"].between(minimo, maximo).all()
  assert ventas_q["seller_state"].isin(estados_obj).all()
  assert (ventas_q["freight_value"] < 0.2 * ventas_q["price"]).all()
  assert n_q == 101, n_q
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Fórmulas con `eval` ⭐⭐

Con **un solo `eval`** multilínea sobre `products` crea:

- `volumen_l` = largo × alto × ancho / 1000 (litros)
- `peso_kg` = peso / 1000
- `peso_cubicado_kg` = `volumen_l` / 6 (regla típica de paquetería: 6 L ≈ 1 kg)
- `peso_facturable_kg` = el **mayor** entre `peso_kg` y `peso_cubicado_kg`

Pista: `eval` no tiene `max` por fila, pero sí admite comparaciones y aritmética: `a + (b - a) * (b > a)` es el máximo de `a` y `b`. Guarda el resultado en `prod_e3`.

Después: `pct_cubicado`: % de productos (con datos) donde el peso **cubicado** es mayor que el real.

In [ ]:
# TU CÓDIGO AQUÍ
prod_e3 = ...
pct_cubicado = ...

In [ ]:
# Comprobación
if pendiente(prod_e3, pct_cubicado):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert {"volumen_l", "peso_kg", "peso_cubicado_kg", "peso_facturable_kg"} <= set(prod_e3.columns)
  ok = prod_e3.dropna(subset=["peso_kg", "peso_cubicado_kg"])
  assert np.allclose(ok["peso_facturable_kg"], np.maximum(ok["peso_kg"], ok["peso_cubicado_kg"]))
  assert np.isclose(pct_cubicado, 66.2162, atol=0.01), pct_cubicado
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · Etapas con `pipe` ⭐⭐

Escribe tres funciones y úsalas en una cadena:

- `con_categoria(df)`: agrega a un DataFrame de ítems la columna `categoria` (en inglés; `"sin_categoria"` si falta).
- `con_estado_vendedor(df)`: agrega `seller_state`.
- `ranking(df, llave, valor, n)`: suma `valor` por `llave`, devuelve las `n` filas mayores como DataFrame con columnas `[llave, valor, "pct"]`, donde `pct` es el % sobre el total **de todas** las llaves (no sólo las `n`).

Luego:

1. `top_cat`: `items` → `con_categoria` → `ranking(llave="categoria", valor="price", n=5)`.
2. `top_cat_sp`: igual pero sólo con vendedores de SP (usa `con_estado_vendedor` y un `query` en la cadena).

In [ ]:
# TU CÓDIGO AQUÍ
def con_categoria(df):
  ...

def con_estado_vendedor(df):
  ...

def ranking(df, llave, valor, n=5):
  ...

top_cat = ...
top_cat_sp = ...

In [ ]:
# Comprobación
if pendiente(top_cat, top_cat_sp):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(top_cat.columns) == ["categoria", "price", "pct"] and len(top_cat) == 5
  assert top_cat["price"].is_monotonic_decreasing
  assert top_cat.iloc[0]["categoria"] == "watches_gifts"
  assert np.isclose(top_cat["pct"].iloc[0], 9.0403, atol=0.01), top_cat["pct"].iloc[0]
  assert top_cat_sp.iloc[0]["categoria"] == "bed_bath_table", top_cat_sp.iloc[0]["categoria"]
  assert top_cat_sp["pct"].sum() < 100
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: *scorecard* de vendedores ⭐⭐⭐

Construye `scorecard` con **un pipeline de etapas** (funciones decoradas con `@registrar` y compuestas con `pipe`), una fila por vendedor con:

| columna | definición |
|---|---|
| `seller_state` | estado del vendedor |
| `pedidos` | pedidos **distintos** |
| `ingreso` | suma de `price` |
| `score_medio` | promedio de `review_score` de sus pedidos (usa **una** reseña por pedido: la más reciente) |
| `pct_tarde` | % de sus ítems entregados después de la fecha estimada (sobre ítems con entrega) |
| `rank_ingreso` | ranking por `ingreso` **dentro de su estado** (1 = mayor, `method="min"`) |

Conserva sólo vendedores con **al menos 2 pedidos**, ordena por `ingreso` descendente y deja `seller_id` como índice.

Pistas: una etapa para llevar los ítems a nivel "ítem con datos del pedido", otra para la reseña, otra para agregar por vendedor y otra para el ranking.

In [ ]:
# TU CÓDIGO AQUÍ
scorecard = ...

In [ ]:
# Comprobación
if pendiente(scorecard):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert scorecard.index.name == "seller_id"
  assert list(scorecard.columns) == ["seller_state", "pedidos", "ingreso", "score_medio", "pct_tarde", "rank_ingreso"]
  assert (scorecard["pedidos"] >= 2).all() and scorecard["ingreso"].is_monotonic_decreasing
  assert len(scorecard) == 45, len(scorecard)
  top = scorecard.iloc[0]
  assert np.isclose(top["ingreso"], 907.75), top["ingreso"]
  assert top["rank_ingreso"] == 1
  assert np.isclose(scorecard["score_medio"].mean(), 4.076764, atol=0.001), scorecard["score_medio"].mean()
  print("✅ Ejercicio 5 correcto")

---
# ✅ Soluciones

### Solución 1

In [ ]:
res_e1 = (
  orders
  .query("order_status == 'delivered'")
  .assign(
    anio=lambda d: d["order_purchase_timestamp"].dt.year,
    dias=lambda d: (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days,
  )
  .dropna(subset=["dias"])
  .merge(customers, on="customer_id", validate="1:1")
  .groupby(["anio", "customer_state"], observed=True, as_index=False)["dias"].mean()
  .query("customer_state in ['SP', 'RJ', 'MG']")
  .sort_values(["anio", "dias"], ascending=[True, False])
  .reset_index(drop=True)
)
res_e1

### Solución 2

In [ ]:
ventas_q = (
  items
  .assign(**{"valor total": lambda d: d["price"] + d["freight_value"]})
  .merge(sellers, on="seller_id", validate="m:1")
  .query("@minimo <= `valor total` <= @maximo "
         "and seller_state in @estados_obj "
         "and freight_value < 0.2 * price")
)
n_q = len(ventas_q)
n_q

*Nota:* `query` acepta comparaciones encadenadas (`a <= x <= b`), igual que Python. Partir el string en varias líneas (concatenación implícita de strings entre paréntesis) mantiene la condición legible.

### Solución 3

In [ ]:
prod_e3 = products.eval('''
    volumen_l = product_length_cm * product_height_cm * product_width_cm / 1000
    peso_kg = product_weight_g / 1000
    peso_cubicado_kg = volumen_l / 6
    peso_facturable_kg = peso_kg + (peso_cubicado_kg - peso_kg) * (peso_cubicado_kg > peso_kg)
''')

pct_cubicado = (
  prod_e3.dropna(subset=["peso_kg", "peso_cubicado_kg"])
         .eval("peso_cubicado_kg > peso_kg")
         .mean() * 100
)
round(pct_cubicado, 2)

*Nota:* en `eval`, un booleano se comporta como 0/1 en aritmética, por eso el truco del máximo funciona. Fuera de `eval`, lo natural sería `np.maximum(a, b)` dentro de un `assign`.

### Solución 4

In [ ]:
def con_categoria(df):
  cat_en = (products.set_index("product_id")["product_category_name"]
                    .map(traduccion.set_index("product_category_name")["product_category_name_english"]))
  return df.assign(categoria=lambda d: d["product_id"].map(cat_en).fillna("sin_categoria"))


def con_estado_vendedor(df):
  return df.merge(sellers, on="seller_id", how="left", validate="m:1")


def ranking(df, llave, valor, n=5):
  return (df.groupby(llave, as_index=False, observed=True)[valor].sum()
            .assign(pct=lambda d: d[valor] / d[valor].sum() * 100)
            .nlargest(n, valor)
            .reset_index(drop=True))


top_cat = items.pipe(con_categoria).pipe(ranking, llave="categoria", valor="price", n=5)

top_cat_sp = (items
              .pipe(con_categoria)
              .pipe(con_estado_vendedor)
              .query("seller_state == 'SP'")
              .pipe(ranking, llave="categoria", valor="price", n=5))
top_cat.round(2), top_cat_sp.round(2)

*Nota:* en `ranking`, `pct` se calcula **antes** de `nlargest`, así el denominador es el total de todas las llaves. Si lo hicieras después, los 5 sumarían 100%.

### Solución 5

In [ ]:
@registrar
def items_con_pedido(items_df, orders_df):
  return items_df.merge(
    orders_df[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date"]],
    on="order_id", how="left", validate="m:1",
  ).assign(
    tarde=lambda d: (d["order_delivered_customer_date"] > d["order_estimated_delivery_date"])
                    .astype("float")                                        # 1.0 / 0.0
                    .where(d["order_delivered_customer_date"].notna())     # NaN si no hubo entrega
  )


@registrar
def con_ultima_resena(df, reviews_df):
  ultima = (reviews_df.sort_values("review_answer_timestamp")
                      .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
  return df.merge(ultima, on="order_id", how="left", validate="m:1")


@registrar
def por_vendedor(df):
  # score: un voto por pedido (no por ítem) -> primero reducimos a pedido
  score_ped = (df.drop_duplicates(["seller_id", "order_id"])
                 .groupby("seller_id")["review_score"].mean())
  return (df.groupby("seller_id")
            .agg(pedidos=("order_id", "nunique"),
                 ingreso=("price", "sum"),
                 pct_tarde=("tarde", "mean"))
            .assign(score_medio=score_ped,
                    pct_tarde=lambda d: d["pct_tarde"] * 100))


@registrar
def con_ranking_estado(df, sellers_df):
  return (df.join(sellers_df.set_index("seller_id"))
            .assign(rank_ingreso=lambda d: d.groupby("seller_state", observed=True)["ingreso"]
                                            .rank(ascending=False, method="min")))


scorecard = (
  items
  .pipe(items_con_pedido, orders)
  .pipe(con_ultima_resena, reviews)
  .pipe(por_vendedor)
  .pipe(con_ranking_estado, sellers)
  .query("pedidos >= 2")
  .sort_values("ingreso", ascending=False)
  [["seller_state", "pedidos", "ingreso", "score_medio", "pct_tarde", "rank_ingreso"]]
  .rename_axis("seller_id")
)
scorecard.round(2).head()

*Notas clave:*

- `.astype("float").where(entregado.notna())` deja `tarde` en `NaN` para ítems sin entrega, así `mean` calcula el % **sólo sobre los entregados**, como pide la definición. Convertimos a `float` **antes** del `where`: un booleano con `NaN` se volvería `object`, más lento y con resultados poco claros.
- El score se promedia **por pedido**: si un pedido tiene 3 ítems del mismo vendedor, su reseña no debe pesar triple. Es una decisión de definición que conviene hacer explícita.
- `assign(score_medio=score_ped)` alinea por índice (`seller_id`): aquí sí queremos usar una Series externa, porque está indexada por la misma llave.
- Con el decorador ves la forma de los datos en cada etapa: si un `merge` explotara filas, saltaría a la vista.

---
## Resumen del módulo

- Encadena con paréntesis exteriores: una línea, un paso.
- `assign(col=lambda d: ...)`: la lambda recibe el DataFrame en ese punto; los argumentos se evalúan en orden.
- `query` con `@variable`, `` `columna con espacios` ``, `in`, rangos encadenados y métodos `.str`/`.dt`/`.isna()`.
- `eval` para fórmulas legibles y multi-columna; no asumas que es más rápido: mide.
- `pipe(f, ...)` para insertar funciones propias; tupla `(f, "arg")` si el DataFrame no es el primer argumento.
- Depura con un `pipe` que imprime y devuelve el DataFrame intacto, o con un decorador.
- Pipelines = funciones pequeñas, puras y probables, compuestas con `pipe`.

**Siguiente:** Módulo 09 · Rendimiento: vectorización, Copy-on-Write, numba, Parquet y lectura por chunks.